In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# Load PHQ-9 Initial Collection dataset
dataset = load_dataset("darssanle/PHQ-9-Initial-Collection", split="train")

# Convert to DataFrame
df = pd.DataFrame(dataset)

# Hiển thị cấu trúc
print(f"Total samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Xem một sample
print("\nSample post_text:")
print(df.iloc[0]['post_text'][:500])
print("\nSample annotations:")
print(df.iloc[0]['annotations'])

c:\Users\ADMIN\miniforge3\envs\dnn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total samples: 2003
Columns: ['post_title', 'post_text', 'annotations']

Sample post_text:
When I was in high school a few years back, I was one of the highest competitors in my school. I joined the high school band in freshman year and by senior year I became one of the best in my section. My academics were always straight and I exercised daily. Senior year I enlisted in the military and now I believe it was one of my worst decisions in life. Before I went to boot camp I was motivated, a patriot and believed that the elite joined the military. In senior year I never applied for any s

Sample annotations:
[['Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down', 'yes'], ['Feeling-down-depressed-or-hopeless', 'no'], ['Feeling-tired-or-having-little-energy', 'yes'], ['Little-interest-or-pleasure-in-doing ', 'yes'], ['Moving-or-speaking-so-slowly-that-other-people-could-have-noticed-Or-the-opposite-being-so-fidgety-or-restless-that-you-have-been-mo

In [2]:
df['annotations'][0]

[['Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down',
  'yes'],
 ['Feeling-down-depressed-or-hopeless', 'no'],
 ['Feeling-tired-or-having-little-energy', 'yes'],
 ['Little-interest-or-pleasure-in-doing ', 'yes'],
 ['Moving-or-speaking-so-slowly-that-other-people-could-have-noticed-Or-the-opposite-being-so-fidgety-or-restless-that-you-have-been-moving-around-a-lot-more-than-usual',
  'no'],
 ['Poor-appetite-or-overeating', 'no'],
 ['Thoughts-that-you-would-be-better-off-dead-or-of-hurting-yourself-in-some-way',
  'no'],
 ['Trouble-concentrating-on-things-such-as-reading-the-newspaper-or-watching-television',
  'no'],
 ['Trouble-falling-or-staying-asleep-or-sleeping-too-much', 'no']]

In [3]:
# Mapping từ text gốc trong dataset sang item number chuẩn PHQ-9
TEXT_TO_ITEM = {
    "Little-interest-or-pleasure-in-doing ": 1,
    "Feeling-down-depressed-or-hopeless": 2,
    "Trouble-falling-or-staying-asleep-or-sleeping-too-much": 3,
    "Feeling-tired-or-having-little-energy": 4,
    "Poor-appetite-or-overeating": 5,
    "Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down": 6,
    "Trouble-concentrating-on-things-such-as-reading-the-newspaper-or-watching-television": 7,
    "Moving-or-speaking-so-slowly-that-other-people-could-have-noticed-Or-the-opposite-being-so-fidgety-or-restless-that-you-have-been-moving-around-a-lot-more-than-usual": 8,
    "Thoughts-that-you-would-be-better-off-dead-or-of-hurting-yourself-in-some-way": 9,
}

In [4]:
df['annotations'][0]

[['Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down',
  'yes'],
 ['Feeling-down-depressed-or-hopeless', 'no'],
 ['Feeling-tired-or-having-little-energy', 'yes'],
 ['Little-interest-or-pleasure-in-doing ', 'yes'],
 ['Moving-or-speaking-so-slowly-that-other-people-could-have-noticed-Or-the-opposite-being-so-fidgety-or-restless-that-you-have-been-moving-around-a-lot-more-than-usual',
  'no'],
 ['Poor-appetite-or-overeating', 'no'],
 ['Thoughts-that-you-would-be-better-off-dead-or-of-hurting-yourself-in-some-way',
  'no'],
 ['Trouble-concentrating-on-things-such-as-reading-the-newspaper-or-watching-television',
  'no'],
 ['Trouble-falling-or-staying-asleep-or-sleeping-too-much', 'no']]

In [5]:
import pandas as pd
import numpy as np

def reorder_annotations_to_standard(annotations_list):

    temp_dict = {}
    
    for question_text, answer in annotations_list:
        item_num = TEXT_TO_ITEM.get(question_text)
        
        if item_num is not None:
            temp_dict[item_num] = answer
        else:
            print(f"Warning: Unknown question text: {question_text[:50]}...")
    
    sorted_annotations = []
    for item_num in range(1, 10):
        try: 
        # if item_num in temp_dict:
            sorted_annotations.append([item_num, temp_dict[item_num]])
        except:
        # else:
            sorted_annotations.append([item_num, 'no'])
            print(f"Warning: Missing item {item_num}, defaulting to 'no'")
    
    return sorted_annotations

# Áp dụng cho toàn bộ dataframe
df['sorted_annotations'] = df['annotations'].apply(reorder_annotations_to_standard)

# Kiểm tra kết quả
print("Before sorting (original order):")
print(df.iloc[0]['annotations'][:3])  # 3 items đầu tiên

print("\nAfter sorting (Item 1→9 order):")
print(df.iloc[0]['sorted_annotations'])

Before sorting (original order):
[['Feeling-bad-about-yourself-or-that-you-are-a-failure-or-have-let-yourself-or-your-family-down', 'yes'], ['Feeling-down-depressed-or-hopeless', 'no'], ['Feeling-tired-or-having-little-energy', 'yes']]

After sorting (Item 1→9 order):
[[1, 'yes'], [2, 'no'], [3, 'no'], [4, 'yes'], [5, 'no'], [6, 'yes'], [7, 'no'], [8, 'no'], [9, 'no']]


In [6]:
WEIGHTS = {
    1: 0.93,   # Item 1 - Anhedonia
    2: 1.00,   # Item 2 - Depressed mood (cao nhất)
    3: 0.86,   # Item 3 - Sleep
    4: 0.92,   # Item 4 - Fatigue
    5: 0.86,   # Item 5 - Appetite
    6: 0.99,   # Item 6 - Worthlessness (cao thứ 2)
    7: 0.87,   # Item 7 - Concentration
    8: 0.63,   # Item 8 - Psychomotor (thấp nhất)
    9: 0.96    # Item 9 - Suicidal (cảnh báo đặc biệt)
}

In [7]:
def calculate_weighted_score_from_sorted(sorted_annotations, weights=WEIGHTS):
    """
    Tính điểm PHQ-9 có trọng số từ annotations đã được sắp xếp theo Item 1→9
    
    Args:
        sorted_annotations: list of [item_number, answer] với item_number từ 1-9 theo thứ tự
        weights: dictionary mapping item_number (1-9) to weight value
    
    Returns:
        dict với các loại điểm số khác nhau
    """
    weighted_sum = 0.0
    max_weighted_sum = 9
    item_scores = []
    
    for item_num, answer in sorted_annotations:
        score = 1 if answer.lower() == 'yes' else 0
        weight = weights.get(item_num, 1.0)
        
        weighted_sum += score * weight
        item_scores.append({
            'item': item_num,
            'answer': answer,
            'score': score,
            'weight': weight,
            'contribution': score * weight
        })
    
    # Normalize về thang 0-27 để dễ so sánh với lâm sàng
    normalized_score = (weighted_sum / max_weighted_sum) * 27 if max_weighted_sum > 0 else 0
    
    return {
        'raw_weighted_score': weighted_sum,
        'normalized_score': normalized_score,
        'max_possible_weighted': max_weighted_sum,
        'naive_score': sum(1 for _, ans in sorted_annotations if ans.lower() == 'yes'),
        'item_details': item_scores
    }

In [8]:
# ============================================
# STEP 2.1: Sắp xếp annotations theo chuẩn
# ============================================
df['sorted_annotations'] = df['annotations'].apply(reorder_annotations_to_standard)

# ============================================
# STEP 2.2: Tính các loại điểm số
# ============================================
df['phq9_scores'] = df['sorted_annotations'].apply(calculate_weighted_score_from_sorted)

# ============================================
# STEP 2.3: Extract thành các cột riêng
# ============================================
df['naive_phq9'] = df['phq9_scores'].apply(lambda x: x['naive_score'])
df['weighted_phq9_raw'] = df['phq9_scores'].apply(lambda x: x['raw_weighted_score'])
df['weighted_phq9_norm'] = df['phq9_scores'].apply(lambda x: x['normalized_score'])

# ============================================
# STEP 2.4: Thêm phân loại mức độ
# ============================================
def severity_from_weighted(weighted_norm_score):
    if weighted_norm_score <= 4:
        return "Minimal"
    elif weighted_norm_score <= 9:
        return "Mild"
    elif weighted_norm_score <= 14:
        return "Moderate"
    elif weighted_norm_score <= 19:
        return "Moderately Severe"
    else:
        return "Severe"

df['weighted_severity'] = df['weighted_phq9_norm'].apply(severity_from_weighted)
df['naive_severity'] = df['naive_phq9'].apply(severity_from_weighted)

# ============================================
# STEP 2.5: Display kết quả
# ============================================
print("\nSample Results:")
print(df[['naive_phq9', 'naive_severity', 'weighted_phq9_norm', 'weighted_severity']].head(10))


Sample Results:
   naive_phq9 naive_severity  weighted_phq9_norm  weighted_severity
0           3        Minimal                8.52               Mild
1           3        Minimal                7.86               Mild
2           5           Mild               14.40  Moderately Severe
3           3        Minimal                8.85               Mild
4           2        Minimal                5.97               Mild
5           3        Minimal                8.76               Mild
6           3        Minimal                8.76               Mild
7           3        Minimal                8.76               Mild
8           2        Minimal                5.97               Mild
9           5           Mild               14.40  Moderately Severe


Huấn luyện mô hình

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

# ============================================
# 3.1 Tạo dataset class cho PyTorch
# ============================================
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# ============================================
# 3.2 Chuẩn bị dữ liệu cho classification
# ============================================
# Sử dụng weighted_normalized score để phân loại thành 5 mức độ
severity_mapping = {
    'Minimal': 0,
    'Mild': 1,
    'Moderate': 2,
    'Moderately Severe': 3,
    'Severe': 4
}

df['label'] = df['weighted_severity'].map(severity_mapping)

# Split train/test
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['post_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")

# ============================================
# 3.3 Load MentalBERT và tạo datasets
# ============================================
model_name = "mental/mental-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,  # 5 mức độ trầm cảm
    ignore_mismatched_sizes=True
)

train_dataset = DepressionDataset(train_texts, train_labels, tokenizer)
val_dataset = DepressionDataset(val_texts, val_labels, tokenizer)

# ============================================
# 3.4 Fine-tuning configuration
# ============================================
training_args = TrainingArguments(
    output_dir='./depression_model_results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # Tăng regularization
    weight_decay=0.1,  # Tăng từ 0.01 lên 0.1
    warmup_ratio=0.2,  # Warmup 20% steps
    
    learning_rate=2e-5, 
    warmup_steps=500,
    eval_strategy="steps",
    eval_steps=100,
    logging_dir='./logs',
    logging_steps=100,
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    # Dừng sớm
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

Train samples: 1602
Validation samples: 401
Label distribution:
label
0    126
1    912
2    657
3    290
4     18
Name: count, dtype: int64


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 486.75it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.poo

In [10]:
from transformers import EarlyStoppingCallback
# ============================================
# 3.5 Train model (uncomment để chạy)
# ============================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

# Save model
model.save_pretrained('./mentalbert_phq9_finetuned')
tokenizer.save_pretrained('./mentalbert_phq9_finetuned')

Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
100,1.396378,1.211287,0.451372,0.124399,0.283852
200,1.208365,1.195486,0.458853,0.205714,0.402323
300,1.172083,1.125859,0.508728,0.230681,0.455698
400,1.036561,1.098291,0.528678,0.223297,0.453657


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.32s/it]


('./mentalbert_phq9_finetuned\\tokenizer_config.json',
 './mentalbert_phq9_finetuned\\tokenizer.json')

In [11]:
from typing import List
class TransformerPredictor:
    def __init__(self, model, tokenizer, device='cpu'):
        self.model = model.to(device) 
        self.tokenizer = tokenizer
        self.device = device
        self.class_names = [str(i) for i in range(model.config.num_labels)]

    def predict_proba(self, texts):
        all_probs = []
        
        if isinstance(texts, np.ndarray):
            texts = texts.tolist()
        if isinstance(texts, str):
            texts = [texts]
        texts = [str(t) for t in texts] 
        
        enc = self.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        )
        
        input_ids = enc['input_ids'].to(self.device)
        attn_mask = enc['attention_mask'].to(self.device)
        
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attn_mask)
            probs = torch.nn.functional.softmax(outputs.logits, dim=1).cpu().numpy()
        
        all_probs.append(probs)
        return np.vstack(all_probs)

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return self.predict_proba(list(texts))

In [12]:
import shap
model = AutoModelForSequenceClassification.from_pretrained("./mentalbert_phq9_finetuned")
tokenizer = AutoTokenizer.from_pretrained("./mentalbert_phq9_finetuned")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\n⚙️  Device: {device}")
predictor = TransformerPredictor(model, tokenizer, device=device)

# Masker phải dùng tokenizer của chính model
masker = shap.maskers.Text(tokenizer)

explainer = shap.Explainer(
    predictor,           # callable nhận List[str]
    masker,
    output_names=predictor.class_names
)

idx = 48
text = df.loc[idx, 'post_text']
print(df.loc[idx,'naive_severity'])
shap_values = explainer([text], max_evals=500)  # truyền list, không phải string
shap_values

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 586.85it/s, Materializing param=classifier.weight]                                      



⚙️  Device: cuda
Mild


PartitionExplainer explainer: 2it [00:17, 17.09s/it]               


.values =
array([[[-1.15245261e-04, -2.99317855e-04,  4.53621894e-04,
          2.24492978e-05, -6.15080469e-05],
        [-1.15245261e-04, -2.99317855e-04,  4.53621894e-04,
          2.24492978e-05, -6.15080469e-05],
        [-1.15245261e-04, -2.99317855e-04,  4.53621894e-04,
          2.24492978e-05, -6.15080469e-05],
        ...,
        [-1.25180037e-04, -1.39587745e-03,  1.53073637e-03,
          2.41023954e-05, -3.37816115e-05],
        [-1.25180037e-04, -1.39587745e-03,  1.53073637e-03,
          2.41023954e-05, -3.37816115e-05],
        [-1.25180037e-04, -1.39587745e-03,  1.53073637e-03,
          2.41023954e-05, -3.37816115e-05]]], shape=(1, 508, 5))

.base_values =
array([[0.09510309, 0.32543519, 0.38191333, 0.15844348, 0.0391049 ]])

.data =
(array(['', 'I', "'", 'm ', 'at ', 'a ', 'complete ', 'loss ', 'here',
       '. ', 'I ', '(', '27', 'f', ') ', 'suffer ', 'from ',
       'depression ', 'and ', 'anxiety ', 'myself', '. ', 'I', "'", 'm ',
       'having ', 'trouble ', '

In [13]:
import shap
import numpy as np
import torch
from typing import List, Optional

class SHAPExplainer:
    def __init__(self, model, tokenizer, device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = tokenizer

        self.predictor = TransformerPredictor(model, tokenizer, device=self.device)
        self.class_names = self.predictor.class_names

        # ✅ output_type="str" đảm bảo masker luôn truyền string cho predictor
        masker = shap.maskers.Text(tokenizer, output_type="str")

        self.explainer = shap.Explainer(
            self.predictor,
            masker,
            output_names=self.class_names
        )

    def predict(self, texts: List[str]) -> np.ndarray:
        return self.predictor.predict_proba(texts)

    def explain_text(self, text: str, max_evals: int = 500):
        shap_values = self.explainer([text], max_evals=max_evals)

        probs = self.predictor.predict_proba([text])[0]
        predicted_class = int(np.argmax(probs))

        tokens = self.tokenizer.tokenize(text)
        token_shap_values = shap_values[:, :, predicted_class].values[0]
        token_shap_dict = {
            token: float(val)
            for token, val in zip(tokens, token_shap_values)
        }

        return shap_values, token_shap_dict

    def explain_batch(self, texts: List[str], max_evals: int = 500):
        return self.explainer(texts, max_evals=max_evals)

In [14]:
# Khởi tạo một lần duy nhất
shap_explainer = SHAPExplainer(model, tokenizer)


# Hoặc dùng trực tiếp
# shap_values, token_shap_dict = shap_explainer.explain_text(df.loc[48, 'post_text'])
# probs = shap_explainer.predict("I feel really sad today")

In [15]:
import re

PRONOUNS = {
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself","she","her","hers","herself",
    "it","its","itself","they","them","their","theirs","themselves",
    "who","whom","whose","which","what","this","that","these","those"
}

PREPOSITIONS = {
    "in","on","at","by","for","with","about","against","between",
    "into","through","during","before","after","above","below",
    "to","from","up","down","of","off","over","under","again",
    "further","then","once","out","as","per","via","near","among",
    "within","without","upon","along","behind","beside","beyond",
    "despite","except","inside","outside","since","toward","until",
    "versus","whereas","around","across","throughout","underneath"
}

ARTICLES = {"a", "an", "the"}

STOPWORDS = PRONOUNS | PREPOSITIONS | ARTICLES


def remove_stopwords(text: str) -> tuple[str, list[str]]:
    """
    Loại bỏ đại từ, giới từ, mạo từ khỏi chuỗi văn bản.
    Trả về (văn bản đã lọc, danh sách từ đã xóa).
    """
    tokens = re.findall(r"\b\w+\b|[^\w\s]", text)
    kept, removed = [], []

    for token in tokens:
        if token.lower() in STOPWORDS:
            removed.append(token)
        else:
            kept.append(token)

    return " ".join(kept), removed


# --- Ví dụ ---
text = "She went to the store and he bought some apples for her."
clean, removed = remove_stopwords(text)

print("Gốc:   ", text)
print("Sạch:  ", clean)
print("Đã xóa:", removed)

Gốc:    She went to the store and he bought some apples for her.
Sạch:   went store and bought some apples .
Đã xóa: ['She', 'to', 'the', 'he', 'for', 'her']


In [16]:
import re
import random
import numpy as np
import torch
from transformers import pipeline

# ─────────────────────────────────────────────
# STOPWORD DEFINITIONS
# ─────────────────────────────────────────────
PRONOUNS = {
    "i","me","my","myself","we","our","ours","ourselves",
    "you","your","yours","yourself","yourselves",
    "he","him","his","himself","she","her","hers","herself",
    "it","its","itself","they","them","their","theirs","themselves",
    "who","whom","whose","which","what","this","that","these","those",
    "anyone"
}

PREPOSITIONS = {
    "in","on","at","by","for","with","about","against","between",
    "into","through","during","before","after","above","below",
    "to","from","up","down","of","off","over","under","again",
    "further","then","once","out","as","per","via","near","among",
    "within","without","upon","along","behind","beside","beyond",
    "despite","except","inside","outside","since","toward","until",
    "versus","whereas","around","across","throughout","underneath",
    "else"
}

PUNCTUATION = ["?",".","!"]

ARTICLES = {"a", "an", "the"}

STOPWORDS = PRONOUNS | PREPOSITIONS | ARTICLES


def remove_stopwords(text: str) -> tuple[str, list[str]]:
    """
    Loại bỏ đại từ, giới từ, mạo từ khỏi văn bản.
    Trả về (văn bản đã lọc, danh sách từ đã xóa).
    """
    tokens = re.findall(r"\b\w+\b|[^\w\s]", text)
    kept, removed = [], []
    for token in tokens:
        if token.lower() in STOPWORDS:
            removed.append(token)
        else:
            kept.append(token)
    return " ".join(kept), removed


def is_stopword(token: str) -> bool:
    """Kiểm tra một token (có thể có prefix Ġ của BPE) có phải stopword không."""
    clean = token.strip("Ġ▁").lower()          # Hugging Face BPE prefix
    clean = re.sub(r"[^a-z]", "", clean)        # bỏ ký tự đặc biệt
    return clean in STOPWORDS


# ─────────────────────────────────────────────
# COUNTERFACTUAL GENERATOR
# ─────────────────────────────────────────────
class CounterfactualGenerator:
    def __init__(self, model, tokenizer, shap_explainer):
        self.model = model
        self.tokenizer = tokenizer
        self.shap_explainer = shap_explainer
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.generator = pipeline(
            "text-generation",
            model="distilgpt2",
            max_new_tokens=512
        )

    # ── PUBLIC API ────────────────────────────
    def find_counterfactual(
        self,
        original_text: str,
        target_severity: int = 0,
        max_attempts: int = 10
    ) -> dict:
        """
        Tìm minimal edits để chuyển prediction sang target_severity.

        Args:
            original_text  : text gốc
            target_severity: 0=Minimal, 1=Mild, 2=Moderate,
                             3=Moderately Severe, 4=Severe
            max_attempts   : số lần thử tối đa
        """
        # ── 1. Dự đoán hiện tại ──────────────
        current_probs   = self.shap_explainer.predict([original_text])[0]
        current_class   = int(np.argmax(current_probs))
        current_severity = list(severity_mapping.keys())[current_class]

        if current_class == target_severity:
            return {
                "success"          : True,
                "message"          : "Already at target severity",
                "original_text"    : original_text,
                "current_severity" : current_severity,
            }

        # ── 2. SHAP – lọc stopwords ──────────
        # Xoá stopwords khỏi text trước khi đưa vào SHAP
        filtered_text, removed_sw = remove_stopwords(original_text)

        shap_values, _ = self.shap_explainer.explain_text(filtered_text)
        token_shap      = shap_values[:, :, current_class].values[0]
        tokens          = self.tokenizer.tokenize(filtered_text)

        # ── 3. Chọn candidate – bỏ stopword token còn sót ──
        # (tokenizer BPE đôi khi tách khác với regex → kiểm tra lại)
        negative_words = [
            (token, float(val))
            for token, val in zip(tokens, token_shap)
            if val > 0 and not is_stopword(token)          # ← lọc stopword
        ]
        negative_words.sort(key=lambda x: x[1], reverse=True)

        print(f"Target severity  : {list(severity_mapping.keys())[target_severity]}")
        # print(f"Stopwords removed: {removed_sw}")
        print(f"Top negative words (after stopword filter): "
              f"{[w for w, _ in negative_words[:100]]}")

        # ── 4. Vòng lặp sinh counterfactual ──
        for attempt in range(max_attempts):
            if attempt < 5 and negative_words:
                modified_text = self._replace_negative_words(
                    original_text, negative_words[: attempt + 1]
                )
            else:
                modified_text = self._add_positive_phrases(original_text)

            new_probs  = self.shap_explainer.predict([modified_text])[0]
            new_class  = int(np.argmax(new_probs))

            if new_class == target_severity:
                return {
                    "success"          : True,
                    "original_text"    : original_text,
                    "filtered_text"    : filtered_text,         # text sau khi lọc SW
                    "removed_stopwords": removed_sw,            # stopwords đã xoá
                    "modified_text"    : modified_text,
                    "original_severity": current_severity,
                    "new_severity"     : list(severity_mapping.keys())[new_class],
                    "original_prob"    : float(current_probs[current_class]),
                    "new_prob"         : float(new_probs[new_class]),
                    "changed_words"    : [w for w, _ in negative_words[: attempt + 1]],
                    "attempts"         : attempt + 1,
                }

        return {
            "success"          : False,
            "message"          : f"Could not find counterfactual after {max_attempts} attempts",
            "original_text"    : original_text,
            "current_severity" : current_severity,
        }

    # ── PRIVATE HELPERS ───────────────────────
    def _replace_negative_words(self, text: str, negative_words: list) -> str:
        """Thay thế các từ tiêu cực bằng từ tích cực tương ứng."""
        positive_replacements = {
            "sad"       : "happy",
            "sadness"   : "happiness",
            "hopeless"  : "hopeful",
            "tired"     : "energized",
            "tiring"    : "energizing",
            "worthless" : "valuable",
            "failure"   : "success",
            "depressed" : "content",
            "anxious"   : "calm",
            "lonely"    : "connected",
            "empty"     : "fulfilled",
            "burden"    : "blessing",
        }

        modified = text
        for word, _ in negative_words:
            clean_word = word.strip("Ġ▁").lower()
            if clean_word in positive_replacements:
                modified = re.sub(
                    rf"\b{re.escape(clean_word)}\b",
                    positive_replacements[clean_word],
                    modified,
                    flags=re.IGNORECASE
                )
        return modified

    def _add_positive_phrases(self, text: str) -> str:
        """Thêm cụm từ tích cực vào cuối văn bản."""
        positive_phrases = [
            " However, I'm feeling much better now.",
            " But things are improving day by day.",
            " I'm receiving treatment and starting to feel hopeful.",
            " My therapist has been very helpful.",
            " I see light at the end of the tunnel.",
        ]
        return text + random.choice(positive_phrases)


# ─────────────────────────────────────────────
# DEMO
# ─────────────────────────────────────────────
cf_generator = CounterfactualGenerator(model, tokenizer, shap_explainer)

sample_text = df.iloc[1]["post_text"]

print("\n" + "=" * 60)
print("COUNTERFACTUAL ANALYSIS")
print("=" * 60)

result = cf_generator.find_counterfactual(sample_text, target_severity=0)

if result["success"]:
    print(f"\n✅ COUNTERFACTUAL FOUND!")
    print(f"Stopwords removed : {result.get('removed_stopwords', [])}")
    print(f"Filtered text     : {result.get('filtered_text', '')[:200]}")
    print(f"Severity change   : {result['original_severity']} → {result['new_severity']}")
    print(f"Probability change: {result['original_prob']:.3f} → {result['new_prob']:.3f}")
    print(f"Modified text     :\n{result['modified_text'][:300]}...")
    print(f"Changed words     : {result['changed_words']}")
    print(f"Attempts          : {result['attempts']}")
else:
    print(f"\n❌ {result['message']}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 611.43it/s, Materializing param=transformer.wte.weight]            
GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



COUNTERFACTUAL ANALYSIS


Target severity  : Minimal
Top negative words (after stopword filter): ['diagnosed', 'depression', 'and', 'general', 'nine', 'years', 'ago', 'was', '’', 'm', 'too', 'terrified', 'try', 'anything', 'and', 'have', 'd', 'like', 'try', 'medication', 'but', 'productive', 'earth', 'and', 'would', 'human', 'kind', 'are', 'doing', 'nothing', 'be', 'best', 'if', 'walked', 'hand', 'hand', 'extinction', 'save', 'planet', 'kind', 'hate', '##ful', ')', 'or', '##ised', 'anxiety', 'disorder', ',', 'six', '’', 't', 'have', 'single', 'diagnosed', 'pts', '##d', 'due', 'years', 'later', 'was', 'also', 'friend', 'left', '.', '’', 'finally', 'stopped', 'living', 'denial', 'and', 'diagnosis', ',', 'first', 'overcome', '?', 've', 'isolated', 'so', 'much', 'past', 'few', 'years', 'don', 'struggled', '?', 'helped', 'overly', 'critical', 'and', 'hate', 'remember', 'being', 'deeply', 'sad', '##ful', 'view', 'world', '(', '’', 's']

❌ Could not find counterfactual after 10 attempts


In [17]:
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
from collections import Counter

class RAGPrototypeMatcher:
    """
    FAISS-based prototype matcher (in-memory, không Redis).
    Tương thích với interface của PrototypeMatcher gốc.
    """

    def __init__(self, df=None, text_column='post_text', severity_column='weighted_severity'):
        """
        Args:
            df: pandas DataFrame chứa dữ liệu prototypes.
            text_column: tên cột chứa văn bản.
            severity_column: tên cột chứa mức độ trầm cảm.
        """
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.text_column = text_column
        self.severity_column = severity_column

        self._index = None          # FAISS index
        self._prototypes = []       # list of dict chứa metadata
        self.database = None        # giữ lại để tham khảo (optional)

        if df is not None:
            self.fit(df)

    # ---------- Encoding helpers ----------
    def _encode(self, texts):
        """Encode và L2-normalize (inner product = cosine)."""
        vecs = self.encoder.encode(
            texts,
            batch_size=32,
            show_progress_bar=False,
            normalize_embeddings=True
        )
        return vecs.astype(np.float32)

    # ---------- Build index from dataframe ----------
    def fit(self, df, force_rebuild=False):
        """
        Xây dựng FAISS index từ dataframe.
        Mỗi dòng trở thành một prototype với các trường:
          - text, severity, naive_score, weighted_score (nếu có)
        """
        if not force_rebuild and self._index is not None:
            print("Index đã tồn tại. Dùng force_rebuild=True để tạo lại.")
            return

        print(f"Đang xây dựng index cho {len(df)} prototypes...")
        self.database = df.reset_index(drop=True)  # đảm bảo index liên tục

        texts = self.database[self.text_column].tolist()
        embeddings = self._encode(texts)

        # Tạo FAISS index FlatIP
        self._index = faiss.IndexFlatIP(embeddings.shape[1])
        self._index.add(embeddings)

        # Lưu metadata
        self._prototypes = []
        for idx, row in self.database.iterrows():
            proto = {
                'text': row[self.text_column],
                'severity': row[self.severity_column],
                'naive_score': row.get('naive_phq9', None),
                'weighted_score': row.get('weighted_phq9_norm', None),
                # Có thể thêm các cột khác nếu cần
            }
            self._prototypes.append(proto)

        print(f"✅ Index sẵn sàng: {self._index.ntotal} vectors, dim={embeddings.shape[1]}")

    # ---------- Tìm prototypes (giống interface cũ) ----------
    def find_prototypes(self, query_text, k=5, return_similarities=True):
        """
        Tìm k prototypes gần nhất với query_text.

        Returns:
            list[dict]: mỗi dict chứa rank, similarity, text, severity,
                        naive_score, weighted_score
        """
        if self._index is None:
            raise RuntimeError("Chưa có index. Hãy gọi fit() trước.")

        query_vec = self._encode([query_text])           # (1, D)
        scores, indices = self._index.search(query_vec, k)   # scores[0], indices[0]

        results = []
        for rank, (idx, score) in enumerate(zip(indices[0], scores[0])):
            if idx == -1:
                continue
            proto = self._prototypes[idx]
            results.append({
                'rank': rank + 1,
                'similarity': float(score),
                'text': proto['text'],
                'severity': proto['severity'],
                'naive_score': proto['naive_score'],
                'weighted_score': proto['weighted_score']
            })
        return results

    # ---------- Phân tích prototypes (giống get_prototype_summary) ----------
    def get_prototype_summary(self, query_text, k=3):
        """
        Trả về summary thống kê của top-k prototypes.

        Returns:
            dict: prototypes, severity_distribution, avg_similarity, most_common_severity
        """
        prototypes = self.find_prototypes(query_text, k=k)

        if not prototypes:
            return {
                'prototypes': [],
                'severity_distribution': {},
                'avg_similarity': 0.0,
                'most_common_severity': None
            }

        severity_counts = Counter(p['severity'] for p in prototypes)
        avg_sim = np.mean([p['similarity'] for p in prototypes])

        return {
            'prototypes': prototypes,
            'severity_distribution': dict(severity_counts),
            'avg_similarity': avg_sim,
            'most_common_severity': severity_counts.most_common(1)[0][0] if severity_counts else None
        }

    # ---------- Các method bổ sung hữu ích ----------
    def retrieve(self, query_text, top_k=3, severity_filter=None, score_threshold=0.0):
        """Giống find_prototypes nhưng có thêm filter và threshold."""
        results = self.find_prototypes(query_text, k=top_k*2 if severity_filter else top_k)
        filtered = []
        for r in results:
            if r['similarity'] < score_threshold:
                continue
            if severity_filter and r['severity'] != severity_filter:
                continue
            filtered.append(r)
            if len(filtered) == top_k:
                break
        return filtered

    def stats(self):
        """Thông tin về index."""
        if self._index is None:
            return {"total_vectors": 0}
        return {
            "total_vectors": self._index.ntotal,
            "embed_dim": self._index.d,
            "index_type": type(self._index).__name__
        }

    def flush(self):
        """Xoá index trong bộ nhớ."""
        self._index = None
        self._prototypes = []
        self.database = None
        print("Đã flush index.")

# Sử dụng Prototype Matcher
matcher = RAGPrototypeMatcher(df, text_column='post_text', severity_column='weighted_severity')

# Test với một sample
query_idx = 100
query_text = df.iloc[query_idx]['post_text']
query_severity = df.iloc[query_idx]['weighted_severity']

print(f"\nFINDING PROTOTYPES FOR QUERY:")
print(f"Query severity: {query_severity}")
print(f"Query text preview: {query_text[:150]}...\n")

prototype_summary = matcher.get_prototype_summary(query_text, k=5)

print("PROTOTYPE MATCHING RESULTS:")
print(f"Average similarity: {prototype_summary['avg_similarity']:.3f}")
print(f"Most common severity among prototypes: {prototype_summary['most_common_severity']}")
print(f"Severity distribution: {prototype_summary['severity_distribution']}")
print("\nTOP PROTOTYPES:")
for p in prototype_summary['prototypes'][1:5]:
    print(f"  {p['rank']}. Similarity={p['similarity']:.3f} | Severity={p['severity']} | PHQ-9={p['weighted_score']:.1f}")
    print(f"     Text: {p['text'][:100]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 435.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đang xây dựng index cho 2003 prototypes...
✅ Index sẵn sàng: 2003 vectors, dim=384

FINDING PROTOTYPES FOR QUERY:
Query severity: Mild
Query text preview: I am so alone. Not just lonely, completely alone. There's not one person I can turn to for help. I don't know what I did to deserve this suffering. I ...

PROTOTYPE MATCHING RESULTS:
Average similarity: 0.804
Most common severity among prototypes: Moderate
Severity distribution: {'Mild': 2, 'Moderate': 3}

TOP PROTOTYPES:
  2. Similarity=0.765 | Severity=Moderate | PHQ-9=10.5
     Text: Hi, woman in my late twenties here, feel like I shot myself in the foot at every opportunity. I neve...
  3. Similarity=0.757 | Severity=Moderate | PHQ-9=11.5
     Text: I feel so unwanted and worthless. 

I'm 19 years old and I am extremely isolated. I live in a depres...
  4. Similarity=0.752 | Severity=Mild | PHQ-9=8.8
     Text: I cant fucking do this. I cant fucking do this. Its been so goddamn long since ive actually felt hap...
  5. Similarity=